# 01 · Bronze — dado cru, com rastreabilidade

```
landing (arquivos crus) ──► BRONZE (Iceberg) ──► Silver ──► Gold ─┬─► Postgres ──► Metabase
                                                                  └─► chunks ──► pgvector (próximo módulo)
```

**Regra da Bronze:** guardar como veio. Nada de limpar aqui. Só acrescentamos 3 colunas para saber *de onde* e *quando* veio:
`_ingest_ts`, `_source_file`, `_batch_id`.

**Antes de começar:** `docker compose up -d` (pasta `docker/`) e `pip install -r requirements.txt`.

## Passo 0 · O ambiente está de pé?

In [1]:
import sys, subprocess
r = subprocess.run([sys.executable, "../scripts/verifica_ambiente.py"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)

OK    Java 
OK    PySpark 3.5.3 
OK    MinIO (buckets landing e warehouse) 
OK    Spark 3.5.3 (grava e lÃª Parquet) 

Postgres + pgvector e Metabase: vamos montar juntos nas prÃ³ximas etapas.
Tudo pronto.
ÒXITO: o processo com PID 7984 (processo filho de PID 17528) foi finalizado.
ÒXITO: o processo com PID 17528 (processo filho de PID 3480) foi finalizado.
ÒXITO: o processo com PID 3480 (processo filho de PID 9108) foi finalizado.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

[Stage 0:>                                                          (0 + 2) / 2]

                                                                                



## Passo 0.5 · Baixar as dependências do Spark (só na primeira vez)

O Spark precisa de alguns jars (Iceberg, S3A com o AWS SDK, JDBC). Na **primeira execução** ele os baixa da internet, e isso pode levar **vários minutos**, principalmente com a rede da sala dividida. Os arquivos ficam guardados em cache (pasta `.ivy2`) e, depois disso, o Spark abre rápido.

Rode a célula abaixo **em casa, antes da aula**. Ela abre um processo separado, baixa os jars e termina, sem deixar o Spark aberto neste notebook.

In [2]:
import sys, subprocess

r = subprocess.run([sys.executable, "../scripts/preparar_jars.py"],
                   capture_output=True, text=True)
print(r.stdout[-1500:])      # fim da saída: deve terminar com "Jars baixados e prontos"
print(r.stderr[-1500:])      # se houver erro, a causa aparece aqui

:: loading settings :: url = jar:file:/C:/Users/giova/AppData/Local/Programs/Python/Python310/Lib/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Jars baixados e prontos. Spark 3.5.3
ÒXITO: o processo com PID 3380 (processo filho de PID 2368) foi finalizado.
ÒXITO: o processo com PID 2368 (processo filho de PID 23660) foi finalizado.
ÒXITO: o processo com PID 23660 (processo filho de PID 3268) foi finalizado.

nd org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 284ms :: artifacts dl 20ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12

## Passo 1 · Abrir o Spark (modo local)

O Spark roda **dentro deste Python**, com `local[2]` (2 núcleos) e 2 GB. Na primeira vez ele baixa os jars (Iceberg, S3A, JDBC). O S3A traz junto o AWS SDK, que é grande, então pode levar vários minutos e precisa de internet. Os jars ficam em cache em ~/.ivy2 e nas próximas execuções é rápido.
Para mais folga, `SPARK_CORES` e `SPARK_MEMORY` (variáveis de ambiente antes de abrir o Jupyter).
Spark UI: <http://localhost:4040> enquanto houver um job rodando.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))          # para importar a pasta common/
from common import lake_local as L #, medalhao as M usaremos depois
from pyspark.sql import functions as F
spark = L.get_spark()                              # PySpark local: sem cluster
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

Spark 3.5.3 | master: local[2] | storage: minio


## Passo 2 · Os arquivos chegam na landing (object storage)

In [4]:
n = L.upload_to_landing(L.DATA_DIR / "lote1", "mercadonordeste/lote1")
print(n, "arquivos enviados")
for chave, tam in L.list_landing("mercadonordeste/lote1"):
    print(f"{tam:>10,} bytes  {chave}")

4 arquivos enviados
   529,223 bytes  mercadonordeste/lote1/avaliacoes.csv
   420,891 bytes  mercadonordeste/lote1/clientes.csv
 7,571,082 bytes  mercadonordeste/lote1/pedidos.jsonl
     9,044 bytes  mercadonordeste/lote1/produtos.csv


Abra o console do MinIO (<http://localhost:9001>, com o usuário e a senha do seu `.env`) e veja o bucket **landing**.
É um *object storage*: não há pastas de verdade, só chaves com `/` no nome.
Você achou os objetos no bucket warehouse do MinIO, em lakehouse/bronze/<tabela>/data e metadata.

### Experimento: o que é uma "pasta" aqui?

Vamos pedir ao MinIO as chaves de três formas diferentes e fazer o experimento de **renomear**.
Antes de rodar, pense: se não existem pastas, como o console mostra `mercadonordeste` e `lote1` como pastas?

In [5]:
s3 = L._s3()     # cliente S3 (boto3) que o módulo usa para falar com o MinIO

# 1) "Pastas" são só prefixos: pedimos ao MinIO para agrupar as chaves pelo "/"
resp = s3.list_objects_v2(Bucket="landing", Prefix="mercadonordeste/", Delimiter="/")
print("Prefixos:", [p["Prefix"] for p in resp.get("CommonPrefixes", [])])

# 2) Listar por prefixo
for chave, tam in L.list_landing("mercadonordeste/lote1/"):
    print(f"{tam:>10,} bytes  {chave}")

# 3) Renomear não existe: é COPIAR e APAGAR (numa chave de teste, sem mexer nos dados)
s3.put_object(Bucket="landing", Key="teste/a.txt", Body=b"ola")
s3.copy_object(Bucket="landing", Key="teste/b.txt",
               CopySource={"Bucket": "landing", "Key": "teste/a.txt"})
s3.delete_object(Bucket="landing", Key="teste/a.txt")
print("Depois do 'renomear':", [c for c, _ in L.list_landing("teste/")])
s3.delete_object(Bucket="landing", Key="teste/b.txt")      # limpa o teste

#a “pasta” é só um prefixo, e o MinIO a simula quando pedimos o agrupamento por /. 
# Renomear são duas operações (copiar e apagar). 
# Em um bucket com milhões de objetos, “renomear uma pasta” significa copiar tudo, e isso vai pesar mais adiante, quando falarmos de tabelas.

Prefixos: ['mercadonordeste/lote1/']
   529,223 bytes  mercadonordeste/lote1/avaliacoes.csv
   420,891 bytes  mercadonordeste/lote1/clientes.csv
 7,571,082 bytes  mercadonordeste/lote1/pedidos.jsonl
     9,044 bytes  mercadonordeste/lote1/produtos.csv
Depois do 'renomear': ['teste/b.txt']


{'ResponseMetadata': {'RequestId': '18E19EE6076DDDC0',
  'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
  'HTTPStatusCode': 204,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8',
   'x-amz-request-id': '18E19EE6076DDDC0',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '152',
   'x-ratelimit-remaining': '152',
   'x-xss-protection': '1; mode=block',
   'date': 'Sun, 25 Oct 2026 00:40:37 GMT'},
  'RetryAttempts': 0}}

## Passo 3 · Landing → Bronze

In [6]:
from common import medalhao as M
contagens = M.ingest_lote(spark, "lote1", L.landing_uri)
contagens

{'clientes': 5000, 'produtos': 200, 'pedidos': 30150, 'avaliacoes': 6489}

In [7]:
spark.sql("SHOW TABLES IN lake.bronze").show()
spark.table("lake.bronze.pedidos").printSchema()
spark.table("lake.bronze.pedidos").select("pedido_id","cliente_id","status","itens","_batch_id","_source_file").show(3, truncate=60)

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|   bronze|  clientes|      false|
|   bronze|avaliacoes|      false|
|   bronze|  produtos|      false|
|   bronze|   pedidos|      false|
+---------+----------+-----------+

root
 |-- cliente_id: long (nullable = true)
 |-- data_pedido: string (nullable = true)
 |-- forma_pagamento: string (nullable = true)
 |-- itens: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- preco_unit: double (nullable = true)
 |    |    |-- produto_id: long (nullable = true)
 |    |    |-- qtd: long (nullable = true)
 |-- pedido_id: long (nullable = true)
 |-- status: string (nullable = true)
 |-- valor_frete: double (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)

+---------+----------+---------+--------------------------------------------------+---------+-----------

### Onde está a tabela `pedidos`?

No Spark ela é `lake.bronze.pedidos`. No storage, é um conjunto de **objetos**. Vamos olhar:
`data/` guarda os arquivos Parquet (as linhas) e `metadata/` guarda o que o Iceberg usa para saber quais arquivos formam a tabela em cada versão.
Repare que `_batch_id=lote1` aparece **dentro da chave**: a tabela foi particionada por lote.

In [8]:
def resumo_objetos(prefixo):
    objs = L.list_warehouse(prefixo)
    return {"objetos": len(objs),
            "data (parquet)": sum("/data/" in c for c, _ in objs),
            "metadata": sum("/metadata/" in c for c, _ in objs),
            "KB": round(sum(t for _, t in objs) / 1024)}

for chave, tam in L.list_warehouse("lakehouse/bronze/pedidos")[:10]:
    print(f"{tam:>9,}  {chave}")
print(resumo_objetos("lakehouse/bronze/pedidos"))

# não existe “o arquivo da tabela”. 
# Uma tabela é o catálogo mais os arquivos de dados e de metadados. 
# Quem consulta pelo Spark vê uma tabela, e o storage só enxerga objetos.

  471,776  lakehouse/bronze/pedidos/data/_batch_id=lote1/00000-14-810766d6-11a0-4810-9c8d-46791374bd4b-0-00001.parquet
    7,917  lakehouse/bronze/pedidos/metadata/d9d68d6f-0310-40e9-b8fd-3e3572aceb5d-m0.avro
    4,462  lakehouse/bronze/pedidos/metadata/snap-7905765528617452444-1-d9d68d6f-0310-40e9-b8fd-3e3572aceb5d.avro
    3,826  lakehouse/bronze/pedidos/metadata/v1.metadata.json
        1  lakehouse/bronze/pedidos/metadata/version-hint.text
{'objetos': 5, 'data (parquet)': 1, 'metadata': 4, 'KB': 477}


Repare: `itens` chegou como **lista de structs** (o JSON foi preservado) e os CSVs chegaram como texto. A tipagem é problema da Silver.

## Passo 4 · Idempotência
Rode o Passo 3 de novo. As contagens **não** dobram: a ingestão apaga o lote e regrava. Pipeline que duplica ao reexecutar é pipeline frágil.

In [9]:
prefixo = "lakehouse/bronze/pedidos"

linhas_antes  = spark.table("lake.bronze.pedidos").count()
objetos_antes = resumo_objetos(prefixo)

M.ingest_lote(spark, "lote1", L.landing_uri)          # reexecuta o MESMO lote

linhas_depois  = spark.table("lake.bronze.pedidos").count()
objetos_depois = resumo_objetos(prefixo)

print("Linhas  :", linhas_antes, "->", linhas_depois, "| idempotente:", linhas_antes == linhas_depois)
print("Objetos :", objetos_antes["objetos"], "->", objetos_depois["objetos"])
print("  data     :", objetos_antes["data (parquet)"], "->", objetos_depois["data (parquet)"])
print("  metadata :", objetos_antes["metadata"], "->", objetos_depois["metadata"])
spark.sql("SELECT count(*) AS arquivos_na_versao_atual FROM lake.bronze.pedidos.files").show()

Linhas  : 30150 -> 30150 | idempotente: True
Objetos : 5 -> 12
  data     : 1 -> 2
  metadata : 4 -> 10
+------------------------+
|arquivos_na_versao_atual|
+------------------------+
|                       1|
+------------------------+



### O que aconteceu?

As **linhas** não dobraram (idempotência), mas os **objetos** no storage aumentaram. Por quê?
O storage não edita arquivos: ele só cria objetos novos. O Iceberg registra uma nova versão da tabela e deixa de *apontar* para os arquivos antigos, mas eles continuam lá, guardando o passado. A última linha mostra quantos arquivos fazem parte da versão **atual**, comparando com o total de objetos que o storage guarda.
Isso é o que permite voltar no tempo e, ao mesmo tempo, o motivo de existir limpeza de arquivos antigos.

## Checkpoint
- [ ] `lake.bronze` tem `clientes`, `produtos`, `pedidos`, `avaliacoes`.
- [ ] Reexecutar não duplica.
- [ ] Você achou `data/` e `metadata/` em `lakehouse/bronze/<tabela>` e explica por que os objetos aumentaram ao reexecutar.

➡️ Próximo: `02_silver.ipynb`